# 1. IMPORT LIBRARIES
We will import all the libararies here

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Display plots inside the notebook
%matplotlib inline

# Set a clean plotting style
sns.set_style("whitegrid")

# 2. LOAD DATASET 

In [ ]:
df=pd.read_csv("../Dataset/USA_Housing.csv")

# 3. UNDERSTAND DATASET

In [ ]:
# df.head()
# df.tail()
# df.shape
# df.columns
# df.info()
df.describe()

# 4. CLEANING DATASET

We will now check for missing values in each column

In [ ]:
df.isnull().sum()

### Observation
Since there are no missing values we will check the duplicate values.

In [ ]:
df.duplicated().sum()

In [ ]:
df.dtypes

### observation 
he dataset contains no duplicate rows, indicating that each observation is unique and no duplicate removal is required.  
All numerical features are float64 Address is a string column.

In [ ]:
print("Negative values in Price:", (df["Price"] < 0).sum())
print("Negative values in Area Population:", (df["Area Population"] < 0).sum())

In [ ]:
df=df.drop("Address",axis=1)

In [ ]:
df.info()

# 5. EXPLORATORY DATA ANALYSIS (EDA)

We begin EDA by analyzing the target variable (Price). This helps us understand how house prices are distributed and whether the data is symmetric or skewed.

In [ ]:
# HISTOGRAM FOR PRICE

plt.figure(figsize=(8,5))

sns.histplot(df["Price"] , bins=30, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")

plt.show()

In [ ]:
# HISTOGRAM FOR EACH COLUMN VALUE

plt.figure(figsize=(15,10))

df.hist(bins=30, figsize=(15,10))

plt.tight_layout()
plt.show()

### Observations

- Most numerical features follow an approximately normal (bell-shaped) distribution.
- The target variable (`Price`) is also approximately normally distributed with a slight positive skew.
- `Avg. Area Number of Bedrooms` does not follow a normal distribution and appears clustered because it is a count-based feature.
- No feature shows severe skewness or obvious extreme outliers from the histograms.

In [ ]:
# HEATMAP

plt.figure(figsize=(10,8))

sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")

plt.title("Correlation Heatmap")
plt.show()

### Observations

- `Avg. Area Income` has the strongest positive correlation with `Price` (0.64), indicating that areas with higher average incomes tend to have more expensive houses.
- `Avg. Area House Age` (0.45), `Area Population` (0.41), and `Avg. Area Number of Rooms` (0.34) show moderate positive correlations with the target variable.
- `Avg. Area Number of Bedrooms` has a weak positive correlation (0.17) with `Price`, suggesting it contributes less to predicting house prices on its own.
- Most independent features have correlations close to zero with one another, indicating no significant multicollinearity.
- The highest correlation among independent features is between `Avg. Area Number of Rooms` and `Avg. Area Number of Bedrooms` (0.46), which is moderate and not a concern.

In [ ]:
# SCATTER PLOT

features = [
    "Avg. Area Income",
    "Avg. Area House Age",
    "Avg. Area Number of Rooms",
    "Avg. Area Number of Bedrooms",
    "Area Population"
]

plt.figure(figsize=(15,10))

for i, feature in enumerate(features, 1):
    plt.subplot(2, 3, i)
    sns.scatterplot(x=df[feature], y=df["Price"])
    plt.title(f"{feature} vs Price")

plt.tight_layout()
plt.show()

### Observations

- `Avg. Area Income` shows a strong positive linear relationship with `Price`, making it the most influential feature.
- `Avg. Area House Age`, `Avg. Area Number of Rooms`, and `Area Population` show moderate positive relationships with `Price`.
- `Avg. Area Number of Bedrooms` has a weak relationship with `Price` and displays vertical bands due to many repeated values.
- Most relationships appear approximately linear, suggesting that Linear Regression is a suitable model for this dataset.
- No major outliers or unusual patterns are observed in the scatter plots.

In [ ]:
# OUTLIER DETECTION

plt.figure(figsize=(15,8))

for i, column in enumerate(df.columns, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[column])
    plt.title(column)

plt.tight_layout()
plt.show()

# 6. FEATURE ENGINEERING

Feature Engineering is the process of creating new features or transforming existing ones to improve the performance and interpretability of a machine learning model.

In this project, we will:

- Create a new feature called `rooms_per_bedroom`.
- Categorize `Area Population` into Low, Medium, and High groups.

In [ ]:
df["rooms_per_bedroom"] = (
    df["Avg. Area Number of Rooms"] /
    df["Avg. Area Number of Bedrooms"]
)

df[["Avg. Area Number of Rooms",
    "Avg. Area Number of Bedrooms",
    "rooms_per_bedroom"]].head()

In [ ]:
df["population_category"] = pd.qcut(
    df["Area Population"],
    q=3,
    labels=["Low", "Medium", "High"]
)

df[["Area Population", "population_category"]].head()

df["population_category"].value_counts()

# 7. FEATURE SELECTION

Feature Selection is the process of separating the input features (independent variables) from the target variable (dependent variable).

In this project:

- **Features (X):** All numerical variables used to predict house prices.
- **Target (y):** `Price`

The newly created `rooms_per_bedroom` feature is included in the model because it provides additional numerical information.

The `population_category` feature is excluded because it was created only for comparing the model's prediction performance across different population groups and is not used for training the Linear Regression model.

In [ ]:
X = df.drop(["Price", "population_category"], axis=1)
y = df["Price"]

In [ ]:
# Let's verify that the features and target variable have been separated correctly.

X.head()
y.head()

In [ ]:
print(X.shape)
print(y.shape)

# 8. TRAIN-TEST SPLIT

The dataset is divided into training and testing sets.

- The training set is used to train the machine learning model.
- The testing set is used to evaluate the model's performance on unseen data.

This helps us determine how well the model generalizes to new data and prevents overfitting.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

### Observation

The dataset has been successfully split into training and testing sets.

- Training set: 4000 observations (80%)
- Testing set: 1000 observations (20%)

The training data will be used to train the Linear Regression model, while the testing data will be used to evaluate its performance on unseen data.

# 9. MODEL TRAINING

In this step, we will train a Linear Regression model using the training dataset.

Linear Regression is a supervised machine learning algorithm used to predict continuous numerical values by finding the best-fitting linear relationship between the independent variables (features) and the dependent variable (target).

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

In [ ]:
import joblib

joblib.dump(model, "../Model/house_price_model.pkl")

# 10. MAKE PREDICTIONS

After training the Linear Regression model, we use it to predict house prices for the testing dataset.

The predicted values will later be compared with the actual house prices to evaluate the model's performance.

In [ ]:
predictions = model.predict(X_test)
predictions.shape

In [ ]:
comparison = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": predictions
})

comparison.head()

### Observation

The trained Linear Regression model generated predictions for all 1000 observations in the testing dataset.

A comparison of the actual and predicted house prices shows that the predicted values are generally close to the actual values, indicating that the model has learned the underlying relationship between the features and the target variable.

# 11. MODEL EVALUATION

After training the Linear Regression model, we evaluate its performance using different regression evaluation metrics.

The metrics used are:

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- R² Score (Coefficient of Determination)

These metrics help us understand how accurately the model predicts house prices.

In [ ]:
mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

In [ ]:
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

### Observation

The Linear Regression model achieved an R² score of approximately **0.918**, indicating that it explains about **91.8% of the variation** in house prices.

The MAE and RMSE values indicate that the model predicts house prices with relatively low error compared to the overall price range.

After adding the `rooms_per_bedroom` feature, the evaluation metrics changed only slightly, suggesting that this engineered feature did not significantly improve the model's predictive performance.

# 12. ACTUAL VS PREDICTED VALUES

After evaluating the model using numerical metrics, we visualize its performance by comparing the actual house prices with the predicted house prices.

If the predictions are accurate, the points should lie close to a straight diagonal line, indicating that the predicted values are close to the actual values.

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, predictions ,alpha=0.5)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")

plt.show()

### Observation

The scatter plot shows a strong positive relationship between the actual and predicted house prices. 
Most data points are closely clustered, indicating that the Linear Regression model predicts house prices with good accuracy. 
Although there are small differences between the actual and predicted values, no major deviations or unusual patterns are observed.

# 12. POPULATION CATEGORY COMPARISON

To evaluate whether the Linear Regression model performs equally well across different population levels, the prediction errors are compared for three population categories:

- Low
- Medium
- High

The categories were created earlier using `pd.qcut()`. The Mean Absolute Error (MAE) is calculated separately for each category to determine whether the model performs consistently across different population groups.

In [ ]:
results = X_test.copy()
results["Actual Price"] = y_test
results["Predicted Price"] = predictions
results.head()

In [ ]:
results["Population Category"] = df.loc[X_test.index, "population_category"]
results.head()

## Calculating Prediction Error

To compare the model's performance across population categories, we first calculate the absolute prediction error for each house.

The absolute error is the absolute difference between the actual price and the predicted price.

In [ ]:
results["Absolute Error"] = abs(results["Actual Price"] - results["Predicted Price"])
results.head()

## Comparing Average Prediction Error by Population Category

The Mean Absolute Error (MAE) is calculated separately for each population category (Low, Medium, and High). This helps determine whether the model performs consistently across different population levels.

In [ ]:
category_error = results.groupby("Population Category")["Absolute Error"].mean()

print(category_error)

In [ ]:
category_error.plot(kind="bar")

plt.title("Average Prediction Error by Population Category")
plt.xlabel("Population Category")
plt.ylabel("Mean Absolute Error")
plt.xticks(rotation=0)

plt.show()